# Assignment 6
Develop a Python-based system to analyze product feedback in terms of likes and dislikes using the DGIM algorithm. Follow the steps below:

`Scrape Product Feedback Data`:

Scrape likes and dislikes for at least 5 different products from a website that allows web scraping. If scraping is not feasible, simulate the data.

Represent the feedback as a binary stream where 1 = like and 0 = dislike.


`Implement DGIM Algorithm`:

Adapt the DGIM (**Datar-Gionis-Indyk-Motwani**) algorithm to estimate the number of likes over sliding time windows efficiently.
The system should handle feedback streams dynamically, where new feedback is added, and outdated feedback is discarded based on the time window.
Queries for Validation:

Execute at least 5 different queries to validate the system, such as:
a. Estimate the number of likes in the last 10 intervals.
b. Compare the DGIM estimate with the exact count for a given interval.
c. Evaluate the performance of the algorithm for longer streams (e.g., 50 intervals).
d. Count dislikes over a time window by inverting the binary stream.
e. Test the system on smaller time windows (e.g., last 5 intervals).

Expected Outputs:

The system should output both the exact count and the DGIM estimate for each query. Compare the results to demonstrate the accuracy and efficiency of the DGIM algorithm.

Input Example:

Binary stream for a single product (simulated for 10 intervals):
[1, 0, 1, 1, 0, 1, 1, 0, 1, 1]
Time window: 5 intervals.

Output Example:

Exact count of likes in last 5 intervals: 4  
Estimated count of likes in last 5 intervals: 4  

Exact count of likes in last 10 intervals: 7  
Estimated count of likes in last 10 intervals: 7  
Implement the solution in Python and validate your system with at least 5 queries

## Import Data and Libraries

In [22]:
import pandas as pd
import numpy as np
import time
import random

# Read your dataset (adjust path as needed)
df = pd.read_csv('/kaggle/input/products-feedback/products-feedback.csv')
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
# Inspect the columns
print(df.columns)
df.sample(5)

Index(['timestamp', 'product_id', 'feedback'], dtype='object')


,timestamp,product_id,feedback
91,142,E90,0
146,8,A12,0
149,84,D78,1
199,103,A12,0
110,167,B34,0


## Data Preprocessing and Binary Representation
Not needed though, already in correct format

In [23]:
# Suppose the "feedback" column contains "like"/"dislike" or 1/0 strings.
# If your data is already 0/1, you can skip the mapping step below.

def map_feedback_to_binary(feedback):
    """
    Maps 'like' -> 1, 'dislike' -> 0.
    If your dataset uses different values, adjust accordingly.
    """
    if feedback == 'like':
        return 1
    elif feedback == 'dislike':
        return 0
    else:
        # fallback or additional parsing if needed
        # For example, you might parse "like" or "0/1" from the string
        return int(feedback)  # If feedback is already '0' or '1' string

df['feedback'] = df['feedback'].apply(map_feedback_to_binary)

# Confirm the transformation
df.sample(5)

,timestamp,product_id,feedback
27,68,B34,1
77,140,D78,1
95,195,E90,0
18,83,D78,1
115,124,C56,1


## DGIM Implementation
The Datar-Gionis-Indyk-Motwani (DGIM) algorithm is a stream-based method to estimate the count of 1s (likes) in a sliding window of size N using a logarithmic amount of memory. The core idea is:

1. Maintain “buckets” of 1s.

2. Each bucket has a size (power of 2) and a timestamp indicating the most recent 1 in that bucket.

3. When a new 1 arrives, create a bucket of size 1. If there are more than two buckets of the same size, merge the oldest two into one bucket of the next size (double the size).

4. Buckets older than the window size are discarded.

5. To estimate the number of 1s in the last N intervals, sum up the bucket sizes, with a partial count for the oldest bucket if it straddles the boundary.

In [24]:
from collections import deque
import math

class DGIM:
    def __init__(self, window_size):
        """
        Initialize DGIM with a specified window size.
        Each entry in 'buckets' is a tuple: (timestamp_of_most_recent_1, bucket_size).
        """
        self.window_size = window_size
        self.buckets = deque()  # Using deque for efficient pops from left/right

    def _expire_old_buckets(self, current_time):
        """
        Remove buckets that fall outside the sliding window (older than current_time - window_size).
        """
        cutoff = current_time - self.window_size
        while self.buckets and self.buckets[0][0] <= cutoff:
            self.buckets.popleft()

    def add_feedback(self, bit, current_time):
        """
        Process new feedback bit (0 or 1) at a given timestamp.
        If bit=1, create a new bucket and handle merges if necessary.
        Then expire old buckets.
        """
        # 1) Expire old buckets first
        self._expire_old_buckets(current_time)

        # 2) If new bit is 1, create a new bucket of size 1
        if bit == 1:
            self.buckets.append((current_time, 1))

            # 3) Merge buckets if there are more than 2 of the same size
            self._merge_buckets()

        # 4) Expire old buckets again (optional check after merges)
        self._expire_old_buckets(current_time)

    def _merge_buckets(self):
        """
        Merge buckets if there are more than two of the same size.
        We look from right (newest) to left (oldest).
        """
        # We'll collect buckets in a temporary list to handle merges from the right
        temp = []
        while self.buckets:
            bucket = self.buckets.pop()
            temp.append(bucket)
            # Check if we have at least three buckets of the same size
            if len(temp) >= 3:
                size1 = temp[-1][1]
                size2 = temp[-2][1]
                size3 = temp[-3][1]
                if size1 == size2 == size3:
                    # Merge the two oldest buckets (the last two in temp)
                    # The newest is temp[-1], so we merge temp[-2] and temp[-3].
                    merged_timestamp = temp[-2][0]  # Keep the most recent timestamp of the merge
                    merged_size = size2 * 2
                    # Remove them from temp
                    temp.pop()
                    temp.pop()
                    # Add the merged bucket
                    temp[-1] = (merged_timestamp, merged_size)
        # Now put them back into self.buckets
        while temp:
            self.buckets.append(temp.pop())

    def estimate_count(self, current_time):
        """
        Estimate the count of 1s in the last 'window_size' intervals up to current_time.
        """
        # First, remove expired buckets
        self._expire_old_buckets(current_time)

        total = 0
        # We'll partially count the oldest bucket if it extends beyond the boundary
        for i, (t, size) in enumerate(self.buckets):
            if i == len(self.buckets) - 1:
                # Oldest bucket might be partially included
                # fraction = how much of it is inside the window
                # For simplicity, we assume half the bucket is an upper bound.
                total += size // 2
            else:
                total += size
        return total


## Feeding the stream into DGIM
Simulating using sliding window

In [25]:
# Group data by product_id so each product has its own DGIM instance
product_ids = df['product_id'].unique()

# Create a dictionary of DGIM objects, one per product
window_size = 10  # Example window size; adjust as needed
dgim_dict = {pid: DGIM(window_size=window_size) for pid in product_ids}

# Sort by timestamp to simulate streaming in chronological order
df_sorted = df.sort_values(by='timestamp')

for idx, row in df_sorted.iterrows():
    current_time = row['timestamp']
    product_id = row['product_id']
    bit = row['feedback']  # 0 or 1
    dgim_dict[product_id].add_feedback(bit, current_time)


## Queries and Validation

#### Helper Functions

In [26]:
def exact_count_likes(df, product_id, current_time, window_size):
    """
    Return the exact number of likes in the last 'window_size' intervals 
    up to 'current_time' for a specific product.
    """
    cutoff = current_time - window_size
    subset = df[(df['product_id'] == product_id) &
                (df['timestamp'] > cutoff) &
                (df['timestamp'] <= current_time)]
    return subset['feedback'].sum()  # sum of 1s

def exact_count_dislikes(df, product_id, current_time, window_size):
    """
    Return the exact number of dislikes (0) in the last 'window_size' intervals.
    """
    cutoff = current_time - window_size
    subset = df[(df['product_id'] == product_id) &
                (df['timestamp'] > cutoff) &
                (df['timestamp'] <= current_time)]
    return len(subset) - subset['feedback'].sum()  # total - likes = dislikes


### Running Queries

In [27]:
# Let's pick a product to demonstrate queries. 
# In practice, you can loop through multiple products.
demo_product = product_ids[0]

# Get the latest timestamp from the data to anchor queries
max_timestamp = df_sorted['timestamp'].max()

# 1) Estimate the number of likes in the last 10 intervals
window_1 = 10
dgim_est_1 = dgim_dict[demo_product].estimate_count(max_timestamp)
exact_1 = exact_count_likes(df_sorted, demo_product, max_timestamp, window_1)
print(f"Query 1: Last {window_1} intervals for product {demo_product}")
print(f"  DGIM estimate = {dgim_est_1}")
print(f"  Exact count   = {exact_1}")

# 2) Compare DGIM estimate with exact count for a given interval (say 20)
window_2 = 20
dgim_est_2 = dgim_dict[demo_product].estimate_count(max_timestamp)
exact_2 = exact_count_likes(df_sorted, demo_product, max_timestamp, window_2)
print(f"\nQuery 2: Last {window_2} intervals for product {demo_product}")
print(f"  DGIM estimate = {dgim_est_2}")
print(f"  Exact count   = {exact_2}")

# 3) Evaluate performance for a larger window (e.g., 50 intervals)
#    We'll temporarily create a new DGIM with window_size=50 for demonstration
dgim_50 = DGIM(window_size=50)
# Re-feed the stream for this new DGIM
for idx, row in df_sorted[df_sorted['product_id'] == demo_product].iterrows():
    dgim_50.add_feedback(row['feedback'], row['timestamp'])

window_3 = 50
dgim_est_3 = dgim_50.estimate_count(max_timestamp)
exact_3 = exact_count_likes(df_sorted, demo_product, max_timestamp, window_3)
print(f"\nQuery 3: Last {window_3} intervals for product {demo_product}")
print(f"  DGIM estimate = {dgim_est_3}")
print(f"  Exact count   = {exact_3}")

# 4) Count dislikes by inverting the binary stream
#    We can do exact_count_dislikes vs. (window_size - dgim_est).
dgim_dislikes_est_10 = window_1 - dgim_est_1
exact_dislikes_10 = exact_count_dislikes(df_sorted, demo_product, max_timestamp, window_1)
print(f"\nQuery 4: Last {window_1} intervals DISLIKES for product {demo_product}")
print(f"  DGIM estimate (inverted) = {dgim_dislikes_est_10}")
print(f"  Exact dislikes           = {exact_dislikes_10}")

# 5) Test the system on smaller window (e.g., last 5 intervals)
window_5 = 5
dgim_5 = DGIM(window_size=5)
for idx, row in df_sorted[df_sorted['product_id'] == demo_product].iterrows():
    dgim_5.add_feedback(row['feedback'], row['timestamp'])

dgim_est_5 = dgim_5.estimate_count(max_timestamp)
exact_5 = exact_count_likes(df_sorted, demo_product, max_timestamp, window_5)
print(f"\nQuery 5: Last {window_5} intervals for product {demo_product}")
print(f"  DGIM estimate = {dgim_est_5}")
print(f"  Exact count   = {exact_5}")

Query 1: Last 10 intervals for product E90
  DGIM estimate = 4
  Exact count   = 7

Query 2: Last 20 intervals for product E90
  DGIM estimate = 4
  Exact count   = 7

Query 3: Last 50 intervals for product E90
  DGIM estimate = 4
  Exact count   = 7

Query 4: Last 10 intervals DISLIKES for product E90
  DGIM estimate (inverted) = 6
  Exact dislikes           = 3

Query 5: Last 5 intervals for product E90
  DGIM estimate = 2
  Exact count   = 4
